# Detalhes do projeto: 

- Versão Python: 3.14.7

# 2. DESAFIO
Você acaba de ser contratado como Analista de Dados Júnior em uma empresa de varejo que possui um histórico de vendas em formato CSV e precisa de um relatório analítico para a reunião trimestral da diretoria. O time de negócios quer entender:

- Como as vendas se comportam ao longo do tempo (por mês e por trimestre);
- Quais produtos e categorias geram mais receita?
- Quais regiões têm melhor desempenho?
- Quais clientes são mais valiosos (segmentação simples por faixa de gasto)?
- Quantas vendas individuais tiveram receita acima da média geral de receita por venda?


In [1]:
# Agrupando importações necessárias para o projeto 
import os
import csv
import json
import random
import re
from datetime import datetime, timedelta
from funcoes_dataset import carregar_dataset, inspecionar_dados, limpar_dados, criar_colunas_derivadas

# RF01 – Criar ou Carregar o Dataset de Vendas

Gerando sementes do dataset conforme recomendação do guia para o projeto. (Desenvolvedor(a) em IA para Análise Preditiva [T4])

Dei preferência por modularizar o código, já que não faz parte do escopo principal e não manter o código completo no mesmo arquivo do projeto principal.

In [2]:
# Importando função para gerar as sementes do dataset
from gerar_sementes import gerar_dataset_vendas

# chamando a função
gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42)

Dataset gerado com 200 registros em vendas.csv.


# RF02 – Inspecionar e Descrever os Dados

Observações: O arquivo, ou pelo menos o nome do arquivo e formato precisam existir antes de executar o comando.

Em versões futuras, criar uma função para validar a existência do arquivo antes de importar um arquivo inexistente e informar na tela para o usuário que o arquivo não existe antes de "quebrar" fluxo do código.

Importando o dataset

In [3]:
# carregar_dataset(caminho_csv="vendas.csv")

registros = carregar_dataset("vendas.csv")
inspecionar_dados(registros)


=== INSPECAO INICIAL DO DATASET ===
Total de registros: 200

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Valores ausentes por coluna:
{'id_venda': 0, 'data_venda': 0, 'cliente': 0, 'produto': 0, 'categoria': 0, 'regiao': 0, 'quantidade': 10, 'preco_unitario': 4}

Primeiros registros:
{'id_venda': '1', 'data_venda': '2025-05-21', 'cliente': 'cliente#016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': '2', 'preco_unitario': '102.9'}
{'id_venda': '2', 'data_venda': '2025-09-16', 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}
{'id_venda': '3', 'data_venda': '2025-03-23', 'cliente': 'Cliente_045', 'produto': 'Tablet', 'categoria': 'Celulares', 'regiao': 'Centro-Oeste', 'quantidade': '1', 'preco_unitario': '1939.76'}
{'id_venda': '4', 'data_venda': '2025-11-06', 'cliente': 'Cliente_017', 'p

[{'id_venda': '1',
  'data_venda': '2025-05-21',
  'cliente': 'cliente#016',
  'produto': 'Mouse',
  'categoria': 'Perifericos',
  'regiao': 'Sudeste',
  'quantidade': '2',
  'preco_unitario': '102.9'},
 {'id_venda': '2',
  'data_venda': '2025-09-16',
  'cliente': 'Cliente_039',
  'produto': 'Notebook',
  'categoria': 'Computadores',
  'regiao': 'Norte',
  'quantidade': '',
  'preco_unitario': '3204.57'},
 {'id_venda': '3',
  'data_venda': '2025-03-23',
  'cliente': 'Cliente_045',
  'produto': 'Tablet',
  'categoria': 'Celulares',
  'regiao': 'Centro-Oeste',
  'quantidade': '1',
  'preco_unitario': '1939.76'},
 {'id_venda': '4',
  'data_venda': '2025-11-06',
  'cliente': 'Cliente_017',
  'produto': 'Notebook',
  'categoria': 'Computadores',
  'regiao': 'Norte',
  'quantidade': '6',
  'preco_unitario': '3864.87'},
 {'id_venda': '5',
  'data_venda': '2025-07-05',
  'cliente': 'Cliente_037',
  'produto': 'Tablet',
  'categoria': 'Celulares',
  'regiao': 'Sul',
  'quantidade': '10',
  'pre

# RF03 – Limpar e Tratar os Dados (datetime e regex)

Implemente uma função de limpeza que trate, no mínimo:
- Espaços extras em campos de texto, usando .strip();
- Datas inválidas: converter data_venda com datetime.strptime(...) dentro de um try/except e remover os registros que não puderam ser convertidos;
- Valores ausentes nos campos críticos quantidade e preco_unitario, descartando o registro quando algum deles estiver vazio;
- Tipos de dados: garantir int para quantidade e float para preco_unitario;
Ruído nos nomes de cliente, usando o módulo re: remover caracteres não alfanuméricos, padronizar o formato para - Cliente_NNN e sinalizar os registros fora do padrão;
- Relatório de limpeza: exibir no console quantos registros entraram, quantos foram removidos (por motivo) e quantos permaneceram.

In [4]:
#limpar_dados(registros)
# Salvando os dados limpos em novas variáveis 
registros_limpos, relatorio = limpar_dados(registros)


=== RELATORIO DE LIMPEZA ===
{'iniciais': 200, 'removidos_data': 4, 'removidos_nulos': 13, 'finais': 183}


# ### Verificar e Corrigir!!!!! ###

data_venda está salvando os dados separados por vírgulas, incluiu "datatime.datetime".

De onde veio isso?

R: Não é erro, é tratamento de dados do Python, vai me permitir avançar nas próximas etapas de análise

In [5]:
#converter_data(registros)

#print(registros[3]) # Testando registro "4" a posição 3

In [6]:
# # conversao de data com tratamento de erro
# try:
#     data = datetime.strptime(texto_data, "%Y-%m-%d")
# except ValueError:
#     data = None                       # marca como invalida

# # limpeza de texto com expressao regular
# nome_limpo = re.sub(r"[^A-Za-z0-9_]", "", nome.strip())

# # validacao de formato com padrao compilado
# padrao_cliente = re.compile(r"^Cliente_\d{3}$", flags=re.IGNORECASE)

# RF04 – Criar Colunas Derivadas com Transformações Condicionais

A partir do dataset limpo, crie no mínimo as seguintes colunas:

Coluna |              Como obter

- receita_total:       quantidade × preco_unitario (multiplicação simples por registro)

- mes:                 Componente de mês extraído do objeto datetime com .month

- mes_nome:            Nome do mês em português, a partir de um dicionário de mapeamento

- trimestre:           Trimestre no formato Q1–Q4, calculado a partir do mês com if/elif/else

- ano:                 Ano extraído do objeto datetime com .year

- faixa_receita_item:  Classificação condicional da receita do registro com if/elif/else

In [7]:
# Dicionário de mapeamento de meses
nome_mes = {1:"Janeiro", 2:"Fevereiro", 3:"Março", 4:"Abril", 5:"Maio", 6:"Junho", 7:"Julho", 8:"Agosto",9:"Setembro",10:"Outubro",11:"Novembro",12:"Dezembro",0:"N/A"}

In [8]:
# Imprimindo dados para tratamento
ler_registro = registros_limpos[0]

print(ler_registro)
print()
print(f'=== Dados Limpos do Registro {ler_registro}')
print()
print("ID:", ler_registro["id_venda"])
print("Data:", ler_registro["data_venda"])
print("Ano:", ler_registro["data_venda"].year)
print("Mês:", ler_registro["data_venda"].month)
print("Qtd:", ler_registro["quantidade"])
print("Preço:", ler_registro["preco_unitario"])

{'id_venda': '1', 'data_venda': datetime.datetime(2025, 5, 21, 0, 0), 'cliente': 'cliente016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': 2, 'preco_unitario': 102.9, 'cliente_fora_do_padrao': True}

=== Dados Limpos do Registro {'id_venda': '1', 'data_venda': datetime.datetime(2025, 5, 21, 0, 0), 'cliente': 'cliente016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': 2, 'preco_unitario': 102.9, 'cliente_fora_do_padrao': True}

ID: 1
Data: 2025-05-21 00:00:00
Ano: 2025
Mês: 5
Qtd: 2
Preço: 102.9


In [9]:
# Trecho de apoio – transformação condicional

# mes = data_venda.month
# if receita_total < 500:
#     faixa_receita_item = "Baixo Valor"
# elif receita_total < 5000:
#     faixa_receita_item = "Medio Valor"
# else:
#     faixa_receita_item = "Alto Valor"


In [10]:
# Testando fórmulas antes do looping
rf04_receita_total = registros_limpos[0]["quantidade"] * registros_limpos[0]["preco_unitario"]
rf04_mes = registros[0]["data_venda"].month
#rf04_mes_nome 
#trimestre
rf04_ano = registros_limpos[0]["data_venda"].year

print(rf04_receita_total)
print(rf04_mes)
print(nome_mes[rf04_mes])
# trimestre print(rf04_trimestre)
print(rf04_ano)

205.8
5
Maio
2025


In [11]:
# 1. Executar a transformacao dos dados limpos
registros_enriquecidos = criar_colunas_derivadas(registros_limpos)

# 2. Imprimir os primeiros 3 registros para inspecao e conferencia
print(f"Total de registros transformados: {len(registros_enriquecidos)}\n")
for reg in registros_enriquecidos[:3]:
    print(f"ID: {reg['id_venda']:<3} | Data: {reg['data_venda'].strftime('%d/%m/%Y')} | "
          f"Mês: {reg['mes_nome']} ({reg['trimestre']}) | "
          f"Receita: R$ {reg['receita_total']:>8.2f} | Faixa: {reg['faixa_receita_item']}")

Total de registros transformados: 183

ID: 1   | Data: 21/05/2025 | Mês: Maio (Q2) | Receita: R$   205.80 | Faixa: Baixo Valor
ID: 3   | Data: 23/03/2025 | Mês: Março (Q1) | Receita: R$  1939.76 | Faixa: Medio Valor
ID: 4   | Data: 06/11/2025 | Mês: Novembro (Q4) | Receita: R$ 23189.22 | Faixa: Alto Valor
